# B2-023-generative-models-diffusion — Practice p07 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** generative-adversarial-network

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import copy
import math
import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
REAL = torch.tensor([[1.0, 1.0], [-1.0, 1.0], [-1.0, -1.0], [1.0, -1.0]])
torch.set_num_threads(1)

## Solution

The two helpers follow the pinned contract literally: each begins with its own optimizer's `zero_grad(set_to_none=True)`, the D-step cuts the generator out of the graph with `.detach()`, the G-step lets the gradient flow **through** the discriminator into the generator, and only the corresponding optimizer steps.
No helper draws random numbers; the noise `Z` is an argument.
A `make_setup()` function rebuilds the pinned setup (seed, generator, discriminator, two Adam optimizers) so that the probes can use independent copies.

In [ ]:
def _check_logits(*tensors):
    for x in tensors:
        if not isinstance(x, torch.Tensor) or x.dim() != 2 or x.shape[-1] != 1:
            raise ValueError("logits must be 2-D tensors with last dimension 1")
    if len({x.dtype for x in tensors}) != 1:
        raise ValueError("logits must share one dtype")


def discriminator_loss(real_logits, fake_logits):
    _check_logits(real_logits, fake_logits)
    return (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))


def generator_loss(fake_logits):
    _check_logits(fake_logits)
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


def make_generator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))


def make_discriminator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))


def d_step(G, D, opt_d, real, z):
    opt_d.zero_grad(set_to_none=True)
    real_logits = D(real)
    fake_logits = D(G(z).detach())
    loss = discriminator_loss(real_logits, fake_logits)
    loss.backward()
    opt_d.step()
    return loss.item()


def g_step(G, D, opt_g, z):
    opt_g.zero_grad(set_to_none=True)
    loss = generator_loss(D(G(z)))
    loss.backward()
    opt_g.step()
    return loss.item()


def gan_step(G, D, opt_g, opt_d, real, z):
    d_loss = d_step(G, D, opt_d, real, z)
    g_loss = g_step(G, D, opt_g, z)
    return (d_loss, g_loss)


def d_step_no_detach(G, D, opt_d, real, z):
    """Defective variant for probe 4: the generator stays in the D-step graph."""
    opt_d.zero_grad(set_to_none=True)
    real_logits = D(real)
    fake_logits = D(G(z))
    loss = discriminator_loss(real_logits, fake_logits)
    loss.backward()
    opt_d.step()
    return loss.item()


def make_setup():
    torch.manual_seed(SEED)
    G = make_generator()
    D = make_discriminator()
    opt_g = torch.optim.Adam(G.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    return G, D, opt_g, opt_d


def all_equal(net_a, net_b):
    return all(torch.equal(a, b) for a, b in zip(net_a.parameters(), net_b.parameters()))


Z = torch.randn(4, 2, generator=torch.Generator().manual_seed(SEED))
G, D, opt_g, opt_d = make_setup()

### Probes 1–3

In [ ]:
# Probe 1: the D-step alone
G_before, D_before = copy.deepcopy(G), copy.deepcopy(D)
d_value = d_step(G, D, opt_d, REAL, Z)
probe1_g_unchanged = all_equal(G, G_before)
probe1_g_grads_none = all(p.grad is None for p in G.parameters())
probe1_d_changed = not all_equal(D, D_before)
with torch.no_grad():
    probe1_expected = discriminator_loss(D_before(REAL), D_before(G(Z))).item()

# Probe 2: the G-step
G_before, D_before = copy.deepcopy(G), copy.deepcopy(D)
g_value = g_step(G, D, opt_g, Z)
probe2_d_unchanged = all_equal(D, D_before)
probe2_g_changed = not all_equal(G, G_before)
with torch.no_grad():
    probe2_expected = generator_loss(D(G_before(Z))).item()

# Probe 3: composition on two fresh copies of the pinned setup
G1, D1, opt_g1, opt_d1 = make_setup()
pair_composed = gan_step(G1, D1, opt_g1, opt_d1, REAL, Z)
G2, D2, opt_g2, opt_d2 = make_setup()
pair_by_hand = (d_step(G2, D2, opt_d2, REAL, Z), g_step(G2, D2, opt_g2, Z))
print("probe 1:", d_value, probe1_expected, probe1_g_unchanged, probe1_g_grads_none, probe1_d_changed)
print("probe 2:", g_value, probe2_expected, probe2_d_unchanged, probe2_g_changed)
print("probe 3:", pair_composed, pair_by_hand)

### Probe 4: the missing-detach variant

On a fresh copy, `d_step_no_detach` leaves gradient tensors in the generator's `.grad` fields, yet the generator's weights do not move.
Only the `.grad` probe can detect this defect, because `opt_d` was built from the discriminator's parameters alone, so `opt_d.step()` never writes to generator weights whether or not the graph was cut.
The bug's footprint is therefore entirely in the generator's `.grad` fields (wasted backward work that a pipeline missing `opt_g.zero_grad` would later apply to the generator), which a weight comparison never sees.

In [ ]:
G4, D4, opt_g4, opt_d4 = make_setup()
G4_before = copy.deepcopy(G4)
d_step_no_detach(G4, D4, opt_d4, REAL, Z)
probe4_some_grad = any(p.grad is not None for p in G4.parameters())
probe4_weights_unchanged = all_equal(G4, G4_before)

# Contrast: the correct d_step on a fresh copy leaves every generator grad None
G5, D5, opt_g5, opt_d5 = make_setup()
d_step(G5, D5, opt_d5, REAL, Z)
print("probe 4:", probe4_some_grad, probe4_weights_unchanged, all(p.grad is None for p in G5.parameters()))

### Answer check

In [ ]:
# Probe 1
assert probe1_g_unchanged
assert probe1_g_grads_none
assert probe1_d_changed
assert isinstance(d_value, float) and math.isclose(d_value, probe1_expected, abs_tol=1e-6, rel_tol=1e-09)
# Probe 2
assert probe2_d_unchanged
assert probe2_g_changed
assert isinstance(g_value, float) and math.isclose(g_value, probe2_expected, abs_tol=1e-6, rel_tol=1e-09)
# Probe 3
assert all_equal(G1, G2) and all_equal(D1, D2)
assert pair_composed == pair_by_hand
assert all(isinstance(v, float) for v in pair_composed)
assert math.isclose(pair_composed[0], 1.4075340033, abs_tol=1e-5, rel_tol=1e-09)
assert math.isclose(pair_composed[1], 0.6521407962, abs_tol=1e-5, rel_tol=1e-09)
# Probe 4
assert probe4_some_grad
assert probe4_weights_unchanged
assert all(p.grad is None for p in G5.parameters())